In [19]:
import os
import json
import shutil
import pickle
import numpy as np
import matplotlib.pyplot as plt
import joblib
import seaborn as sns
import argparse
import sys
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Masking, BatchNormalization, Bidirectional, MaxPooling1D, GlobalMaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam
from keras.utils import plot_model
from collections import Counter
from tensorflow.keras import layers, regularizers
from tensorflow.keras.models import Model
import pandas as pd


In [11]:

pose_indices = [0, 15, 16, 17, 18, 19, 20]
hand_indices = [0, 4, 7, 8, 11, 12, 15, 16, 19, 20]

In [10]:
# --- Step 1: Define Transformer Model ---
# This matches the Transformer class from your fine-tuning code
class PositionEmbedding(layers.Layer):
    def __init__(self, config):
        super(PositionEmbedding, self).__init__()
        self.position_embeddings = self.add_weight(
            name="position_embeddings",
            shape=(config.max_position_embeddings, config.hidden_size),
            initializer="random_normal",
            trainable=True,
        )
    
    def call(self, x):
        seq_length = tf.shape(x)[1]
        positions = tf.range(start=0, limit=seq_length, delta=1)
        position_embeddings = tf.gather(self.position_embeddings, positions)
        return x + position_embeddings

class Transformer(Model):
    def __init__(self, config, n_classes):
        super(Transformer, self).__init__()
        self.l1 = layers.Dense(config.hidden_size, activation=None, kernel_regularizer=regularizers.l2(0.001))
        self.embedding = PositionEmbedding(config)
        
        # Transformer layers
        self.transformer_layers = [
            layers.MultiHeadAttention(
                num_heads=config.num_attention_heads, 
                key_dim=config.hidden_size // config.num_attention_heads,
                kernel_regularizer=regularizers.l2(0.001)
            )
            for _ in range(config.num_hidden_layers)
        ]
        
        # Layer normalization
        self.layer_norms = [
            layers.LayerNormalization(epsilon=1e-6)
            for _ in range(config.num_hidden_layers)
        ]
        
        self.l2 = layers.Dense(n_classes, activation=None, kernel_regularizer=regularizers.l2(0.001))
        self.dropout = layers.Dropout(0.3)
        
    def call(self, x, training=False):
        x = self.l1(x)
        x = self.embedding(x)
        
        for i, layer in enumerate(self.transformer_layers):
            x = layer(x, x)  # Self-attention
            x = self.layer_norms[i](x)
        
        x = tf.reduce_max(x, axis=1)  # Global max pooling
        x = self.dropout(x, training=training)
        x = self.l2(x)
        return x

# --- Step 2: Load SSL Dataset ---
def load_landmarks(filenames, num_frames, glossIndex=3):
    """
    Load and preprocess skeletal landmark data from pickle files.
    
    Args:
        filenames (list): List of file paths to pickle files.
        num_frames (int): Number of frames to sample per video (e.g., 30).
        glossIndex (int): Index in file path to extract gloss (default 3 for SSL).
    
    Returns:
        video_data (np.array): Shape [num_videos, num_frames, num_features].
        labels (np.array): Shape [num_videos], gloss labels.
    """
    video_data = []
    labels = []
    
    pose_indices = [0, 15, 16, 17, 18, 19, 20]  # 7 pose landmarks
    hand_indices = [0, 4, 7, 8, 11, 12, 15, 16, 19, 20]  # 10 hand landmarks per hand
    
    for file in filenames:
        gloss = file.split('/')[glossIndex]
        with open(file, 'rb') as f:
            landmarks = pickle.load(f)
            
            frames = []
            total_frames = len(landmarks)
            step = max(1, total_frames // num_frames)
            
            for i in range(num_frames):
                frame_index = i * step if total_frames >= num_frames else i
                frame = landmarks[frame_index] if frame_index < total_frames else None
                
                # Extract landmarks
                pose_points = frame['pose_landmarks'] if frame and frame['pose_landmarks'] else [None] * 33
                left_hand_points = frame['left_hand_landmarks'] if frame and frame['left_hand_landmarks'] else [None] * 21
                right_hand_points = frame['right_hand_landmarks'] if frame and frame['right_hand_landmarks'] else [None] * 21
                
                # Collect specified points
                extracted_points = []
                for idx in pose_indices:
                    point = pose_points[idx] if pose_points[idx] is not None else {'x': 0, 'y': 0}
                    extracted_points.extend([point['x'], point['y']])
                for idx in hand_indices:
                    point = left_hand_points[idx] if left_hand_points[idx] is not None else {'x': 0, 'y': 0}
                    extracted_points.extend([point['x'], point['y']])
                for idx in hand_indices:
                    point = right_hand_points[idx] if right_hand_points[idx] is not None else {'x': 0, 'y': 0}
                    extracted_points.extend([point['x'], point['y']])
                
                frames.append(extracted_points)
            
            video_data.append(frames)
            labels.append(gloss)
    
    return np.array(video_data), np.array(labels)

In [13]:
# --- Step 2: Load SSL Dataset ---
data_directory = '../all_outputs/output_ssl_small/'
ssl_file_names = [os.path.join(root, f) for root, _, files in os.walk(data_directory) for f in files if f.endswith('.pkl')]
print(f"SSL dataset: {len(ssl_file_names)} files found")

num_frames = 30
ssl_video_data, ssl_labels = load_landmarks(ssl_file_names, num_frames, glossIndex=3)
print(f"SSL data shape: {ssl_video_data.shape}, Labels shape: {ssl_labels.shape}")

SSL dataset: 1280 files found
SSL data shape: (1280, 30, 54), Labels shape: (1280,)


In [16]:
# --- Step 3: Set Up Pretrained INCLUDE Model ---
# Configuration matching training setup
class NewConfig:
    input_size = ssl_video_data.shape[2]  # Number of features (e.g., 54)
    hidden_size = 128
    num_hidden_layers = 4
    num_attention_heads = 8
    max_position_embeddings = 30
    num_classes = 240  # Matches training with 240 classes

new_config = NewConfig()

# Initialize and build the Transformer model
pretrained_model = Transformer(new_config, n_classes=new_config.num_classes)
pretrained_model.build(input_shape=(None, new_config.max_position_embeddings, new_config.input_size))

# Load pretrained weights
model_path = '../saved_models/book_13/transformer_240_classes.h5'  # Path from training
try:
    pretrained_model.load_weights(model_path)
    print(f"Loaded pretrained INCLUDE model from {model_path}")
except Exception as e:
    print(f"Error loading model weights: {e}")
    exit(1)

# Load LabelEncoder
label_encoder_path = '../saved_models/book_13/label_encoder_240.pkl'  # Path from training
try:
    label_encoder = joblib.load(label_encoder_path)
    print(f"Loaded LabelEncoder from {label_encoder_path}")
except Exception as e:
    print(f"Error loading LabelEncoder: {e}")
    exit(1)

Loaded pretrained INCLUDE model from ../saved_models/book_13/transformer_240_classes.h5
Loaded LabelEncoder from ../saved_models/book_13/label_encoder_240.pkl


In [20]:
# Get class names
include_class_names = label_encoder.classes_.tolist()

# Validate class names
if len(include_class_names) != new_config.num_classes:
    raise ValueError(f"Number of INCLUDE class names ({len(include_class_names)}) "
                     f"does not match model output ({new_config.num_classes}).")

# --- Step 4: Predict INCLUDE Classes for SSL Instances ---
# Make predictions on SSL data
ssl_predictions = pretrained_model.predict(ssl_video_data)  # Shape: [num_videos, 240]
predicted_include_indices = np.argmax(ssl_predictions, axis=1)  # Index of highest probability

# --- Step 5: Map Predictions to INCLUDE Glosses ---
# Convert indices to glosses
predicted_include_glosses = [include_class_names[idx] for idx in predicted_include_indices]

# --- Step 6: Output Results ---
# Create a DataFrame to store results
results = pd.DataFrame({
    'ssl_gloss': ssl_labels,
    'predicted_include_gloss': predicted_include_glosses
})

# Print results
print("\nMapping SSL Glosses to Similar INCLUDE Glosses:")
print("---------------------------------------------")
for _, row in results.iterrows():
    print(f"SSL Gloss: {row['ssl_gloss']}, Predicted Similar INCLUDE Gloss: {row['predicted_include_gloss']}")
print("---------------------------------------------")

# Save results to CSV
output_file = 'ssl_to_include_mapping.csv'
results.to_csv(output_file, index=False)
print(f"Results saved to {output_file}")

# Summarize predictions per SSL gloss
summary = results.groupby(['ssl_gloss', 'predicted_include_gloss']).size().reset_index(name='count')
print("\nSummary of Predictions:")
print("---------------------------------------------")
for _, row in summary.iterrows():
    print(f"SSL Gloss: {row['ssl_gloss']}, Predicted INCLUDE Gloss: {row['predicted_include_gloss']}, Count: {row['count']}")
print("---------------------------------------------")

40/40 [==============================] - 0s 2ms/step

Mapping SSL Glosses to Similar INCLUDE Glosses:
---------------------------------------------
SSL Gloss: To_You, Predicted Similar INCLUDE Gloss: Bird
SSL Gloss: To_You, Predicted Similar INCLUDE Gloss: President
SSL Gloss: To_You, Predicted Similar INCLUDE Gloss: Chair
SSL Gloss: To_You, Predicted Similar INCLUDE Gloss: Chair
SSL Gloss: To_You, Predicted Similar INCLUDE Gloss: President
SSL Gloss: To_You, Predicted Similar INCLUDE Gloss: fast
SSL Gloss: To_You, Predicted Similar INCLUDE Gloss: Bird
SSL Gloss: To_You, Predicted Similar INCLUDE Gloss: Chair
SSL Gloss: To_You, Predicted Similar INCLUDE Gloss: Waiter
SSL Gloss: To_You, Predicted Similar INCLUDE Gloss: Chair
SSL Gloss: To_You, Predicted Similar INCLUDE Gloss: Yesterday
SSL Gloss: To_You, Predicted Similar INCLUDE Gloss: Chair
SSL Gloss: To_You, Predicted Similar INCLUDE Gloss: Bird
SSL Gloss: To_You, Predicted Similar INCLUDE Gloss: Chair
SSL Gloss: To_You, Predicted Si